# This program explaines how to build a SimpleRNN that predicts a time series that depends on itself as well as many other time series.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import tensorflow as tf

from tensorflow import keras

from pandas import read_csv
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error


We create 5 time series each with 20,000 random numbers in [1, 2].

We will store this data in an array named mts, of shape (20000, 5).

We will skip normalization!

In [ ]:
dTotal=20000;
t=np.linspace(1,dTotal,dTotal)
mts=1+np.random.rand(dTotal,5)   ## input data is a 20000 by 5  array called mts; there are 5 time series each in its own column.

At this point, as in the slides, we decide that our **Window size is 20 and the inputs per time slots is 5.**

This means we have 20 times slots in each Window.

In ecah time slot, we give 1 number from each of the 5 input series.
**This makes my input shape = (20, 5).**

Number of Samples NS = 20000/(20) = 1000. Each series only gives us one number per time-slot!

Make sure you undrestand why it is not 2000/(20*5). This is clearer in the slides. Look at the picture.

**Input Tensor Shape = (1000, 20, 5).**


mts is a 20000 by 5 array.

How do we get the input data in the shape (1000, 20, 5)?

The issue is we want each of the 5 numbers to come from the 5 different columns of mts.

Will a simple 'reshape' work?

Let us do an experiment!

In [ ]:
## This array is similar to mts, except it has less number of rows.
x=np.random.randint(1, 9, (6, 5))  ### create a 4 by 5 array with integers in [1, 8].
print(x)

In [ ]:
x=np.reshape(x,(3,2,5))  ### Break it into 3 arrays of shape (2, 5).
print(x)   ### Check if the order is the way we wanted.


## Hey! It works!!

In [ ]:
x=mts.reshape(1000,20,5)



On each window, we will get 5 outputs and compare them to the 5 numbers from the 4 times seies to generate the loss functions.

This means I need 5 target numbers per window.

I have already re-shaped the input data into 1000 samples.

I just have to get 5 numbers from the 5 different series.

In [ ]:
z=x[:,5,:]
print(z.shape)

In [ ]:
xtrain=x[0:900]
xtest=x[900:1000]
ztrain=z[0:900]
ztest=z[900:1000]

The Model Part

The first argument of SimpleRNN is the Number of Neurons in the first layer of the ANN that is part of this RNN. It is up to you, just like in ANNs. The more difficult the problem is the more neurons we need.

The number of Neurons in the output layer is the number of outputs we decided already. We said 5 outputs per window. So we have to stick to it.

We use the MSE loss function because we are doing regression.

In [ ]:
model = Sequential()
model.add(SimpleRNN(57, input_shape=(20,5), activation='relu'))  ### Number of Neurons in the first layer = 57
model.add(Dense(5, activation='linear'))                        ### Number of Neurons in the output layer = 4
model.compile(loss='mean_squared_error', optimizer='adam')

In [ ]:
model.fit(xtrain, ztrain, epochs=1, batch_size=1, verbose=2)

In [ ]:
xpredicted=model.predict(xtest)

In [ ]:
print(xpredicted.shape)

Hey!

No Errors!!

This is first step in learing!!!

In [ ]:
for i in range(4):
  print(model.get_weights()[i].shape)

In [ ]:
print(model.get_weights()[0])

In [ ]:
print(model.get_weights()[1])


In [ ]:
print(model.get_weights()[2])

In [ ]:
print(model.get_weights()[3])

In [ ]:
print(model.get_weights()[4])

In [ ]:
plt.figure(figsize=(20, 6), dpi=80)
plt.plot(range(100), xpredicted[:100,1])  ### I plot only one of the outputs, otherwise it will be too crowded!
plt.plot(range(100), ztest[:100,1])